# Lab 63: The Gambler

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 7, *The Decision Bias*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-63.ipynb)

**What you will do:** measure how much shorter your "random" predictions are than real coin flips, compare your 100-flip longest run with the published expectation, pool the class's data, and test whether a small language model states the gambler's fallacy correctly.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 63 you first wrote down a predicted sequence of 20 coin flips, then flipped a real coin
20 times and compared the two. A written "random" sequence almost always has fewer long runs
than a genuinely random one — a sequence like HHHHHTTTTT feels "too streaky" to be random, yet
it is exactly as likely as HTHTHTHTHT. In the smartphone extension, you generated 100 real
flips: the book states that a run of 6 or 7 identical outcomes is mathematically expected in 100
flips, but most people predict the longest run will be only 3 or 4. The **gambler's fallacy** is
the belief that a streak makes the opposite outcome "due" — the coin has no memory, and each
flip is independent.

## Record your results

Enter your predicted 20-flip sequence, your actual 20-flip sequence, your 100-flip longest
run, and what you guessed the longest run would be before flipping.

In [ ]:
from itertools import groupby

def longest_run(sequence):
    return max((len(list(g)) for _, g in groupby(sequence)), default=0)

predicted_seq = "HTHTHHTHTHTHTHTHTHTH"        # example data: replace with your own prediction
actual_seq =    "HHHTTHHHHTTHTTTHHTTH"        # example data: replace with your own coin flips
my_100flip_longest_run = 5                    # example data: from the smartphone extension
my_naive_guess = 4                            # example data: your guess before flipping

seq_results = pd.DataFrame({
    "sequence": ["predicted (by hand)", "actual (coin flips)"],
    "longest_run": [longest_run(predicted_seq), longest_run(actual_seq)],
})
display(seq_results)

## Compare

The book states that in 100 genuine flips, a run of **6 or 7** is mathematically expected,
while most people predict only **3 or 4**.

In [ ]:
print(f"Your predicted 20-flip sequence, longest run: {seq_results.iloc[0].longest_run}")
print(f"Your actual 20-flip sequence, longest run:    {seq_results.iloc[1].longest_run}")
if seq_results.iloc[0].longest_run <= seq_results.iloc[1].longest_run:
    print("Your hand-written sequence had a run no longer than your real coin flips -- people")
    print("faking randomness typically under-produce long runs, exactly as the book describes.")
else:
    print("Your hand-written sequence happened to have a longer run than your real flips this")
    print("time -- with only 20 flips this can happen, but it runs against the usual pattern.")
print()
print(f"Your 100-flip longest run: {my_100flip_longest_run}   |   published expectation: 6-7")
print(f"Your naive guess before flipping: {my_naive_guess}   |   published typical guess: 3-4")

## The AI side

The book frames a coin as the simplest **Markov chain**: the next outcome depends only on the
current state, not on the history, so P(heads) = 0.5 regardless of how long the current streak
is. Below, a large simulation checks this directly by measuring the probability of heads after
streaks of different lengths (the same claim Figure 7.4b illustrates). Then a small open
language model (flan-t5-small) is asked the book's own gambler's-fallacy question, to see
whether it states the correct, memoryless answer.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
rng = np.random.default_rng(2026)
flips = rng.integers(0, 2, 2_000_000)  # 0 = tails, 1 = heads

rows = []
for streak_len in range(1, 7):
    # find positions where the previous `streak_len` flips were all heads
    is_heads = flips == 1
    window = np.lib.stride_tricks.sliding_window_view(is_heads[:-1], streak_len)
    streak_ends = np.all(window, axis=1)
    streak_ends = np.where(streak_ends)[0] + streak_len  # index of the flip right after the streak
    streak_ends = streak_ends[streak_ends < len(flips)]
    p_heads_next = flips[streak_ends].mean() if len(streak_ends) else float("nan")
    rows.append({"preceding streak of heads": streak_len, "P(next flip is heads)": round(p_heads_next, 3),
                 "n streaks observed": len(streak_ends)})
display(pd.DataFrame(rows))

from transformers import pipeline
qa = pipeline("text2text-generation", model="google/flan-t5-small")
prompt = ("I have flipped a fair coin heads five times in a row. Is tails more likely than "
          "heads on the next flip? Answer yes or no and explain briefly.")
print("Model's answer:", qa(prompt, max_new_tokens=40)[0]["generated_text"])

The simulation should show P(next flip is heads) staying close to 0.50 no matter how long the
preceding streak of heads was -- the coin genuinely has no memory. Whether flan-t5-small gives
the correct "no, each flip is independent" answer is a separate question from whether the
simulation is correct: a small model can get a well-known textbook question right by having
seen it during training, without doing any of the reasoning itself, so a correct answer here
is not strong evidence that the model "understands" independence in general.

## Pool the class data

Pool the class's 100-flip longest runs and pre-flip guesses (anonymous IDs, not names).

In [ ]:
import io
csv_text = """id,longest_run_100flips,naive_guess
P01,6,4
P02,5,3
P03,7,4
P04,4,3
P05,8,5
P06,6,4
"""  # example data: replace with your class CSV, e.g. pd.read_csv("gambler_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
display(df.describe()[["longest_run_100flips", "naive_guess"]])
boot_actual = [df.longest_run_100flips.sample(frac=1, replace=True).mean() for _ in range(5000)]
boot_guess = [df.naive_guess.sample(frac=1, replace=True).mean() for _ in range(5000)]
for label, boot in [("actual longest run", boot_actual), ("naive guess", boot_guess)]:
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"Class mean {label}: {np.mean(boot):.1f} (95% bootstrap interval {lo:.1f}-{hi:.1f})")
print("Published values for comparison: actual longest run ~6-7, typical naive guess ~3-4.")

## Questions to think about

1. Was your hand-written "random" sequence's longest run shorter than your real coin flips' longest run? Why might your brain avoid writing down long runs even when trying to be random?
2. How close was your 100-flip longest run to the published expectation of 6 or 7? How close was your pre-flip guess to the published typical guess of 3 or 4?
3. In the Markov-chain simulation, did P(next flip is heads) stay close to 0.50 for every streak length you tested? What would it mean, physically, if it had not?
4. The book notes that the "hot-hand fallacy" debate has been more complicated than first thought (Miller & Sanjurjo, 2018, found a subtle statistical bias in the original debunking). Does knowing that change how confident you feel dismissing a friend's claim that they are "on a streak" at a game?

## Challenge

Ask a friend to type a "random" sequence of 100 H's and T's by hand, without any device, and find
its longest run. Compare it with a genuinely random 100-flip sequence generated by code.

In [ ]:
from itertools import groupby

def longest_run(sequence):
    return max((len(list(g)) for _, g in groupby(sequence)), default=0)

# Example data: replace with your friend's hand-typed sequence.
hand_typed = "HTHTHHTHTHTHHTHTHTHTHTHHTHTHTHTHTHTHTHTHHTHTHTHTHTHTHTHTHTHTHHTHTHTHTHTHTHTHTHTHTHTHTHTHTHTHTHTHTHT"

rng = np.random.default_rng(2026)
machine_generated = "".join(rng.choice(["H", "T"], size=100))

print(f"Hand-typed longest run:        {longest_run(hand_typed)}")
print(f"Machine-generated longest run: {longest_run(machine_generated)}")
print("People faking randomness almost always produce a shorter longest run than a real random")
print("sequence -- the same intuition, working in reverse, that produces the gambler's fallacy.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-63.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")